# NYT Election Article Scraper
Scrapes full article body text from NYT URLs sourced from GDELT.
Covers the 2016, 2020, and 2024 US elections (election day + 2 days after).
Each election saves to its own CSV. Toggle which elections to run in **Section 2**.

## 0. Install dependencies (run once)
```bash
pip install selenium beautifulsoup4 pandas python-dotenv
```

In [1]:
pip install selenium beautifulsoup4 pandas python-dotenv undetected-chromedriver

Note: you may need to restart the kernel to use updated packages.


## 1. Credentials
Create a `.env` file in the same folder as this notebook with:
```
NYT_EMAIL=your@email.com
NYT_PASSWORD=yourpassword
```
The `.env` file is never committed to version control — add it to `.gitignore`.

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env in the current directory

NYT_EMAIL    = os.getenv("NYT_EMAIL")
NYT_PASSWORD = os.getenv("NYT_PASSWORD")

assert NYT_EMAIL and NYT_PASSWORD, (
    "Credentials not found. Create a .env file with NYT_EMAIL and NYT_PASSWORD."
)

## 2. Select elections to scrape
Set each flag to `True` / `False`. 

In [4]:
# ── Toggle elections ──────────────────────────────────────────────────
RUN_2016 = False
RUN_2020 = True
RUN_2024 = True

# ── GDELT CSV paths (SQLDATE | SOURCEURL | AvgTone) ───────────────────
CSV_PATHS = {
    2016: "US2016_cleaned.csv",
    2020: "us2020_cleaned.csv",
    2024: "us2024_cleaned.csv",
}

# ── Output paths ──────────────────────────────────────────────────────
OUTPUT_PATHS = {
    2016: "nyt_2016_election_articles.csv",
    2020: "nyt_2020_election_articles.csv",
    2024: "nyt_2024_election_articles.csv",
}

# ── Election date windows (election day + 2 days after) ───────────────
# GDELT uses integer dates: YYYYMMDD
DATE_WINDOWS = {
    2016: (20161108, 20161110),
    2020: (20201103, 20201105),
    2024: (20241105, 20241107),
}

ELECTIONS_TO_RUN = [
    year for year, flag in [(2016, RUN_2016), (2020, RUN_2020), (2024, RUN_2024)]
    if flag
]
print("Elections queued:", ELECTIONS_TO_RUN)

Elections queued: [2020, 2024]


## 3. Load & filter GDELT CSVs
Keeps NYT URLs only and restricts to the election date window.

In [5]:
import pandas as pd

def load_gdelt(year: int) -> pd.DataFrame:
    path = CSV_PATHS[year]
    start, end = DATE_WINDOWS[year]

    df = pd.read_csv(path)

    # Normalise column names defensively
    df.columns = [c.strip() for c in df.columns]

    # Filter: NYT only
    df = df[df["SOURCEURL"].str.contains("nytimes.com", na=False)].copy()

    # Filter: date window
    df = df[(df["SQLDATE"] >= start) & (df["SQLDATE"] <= end)].copy()

    # Drop duplicates on URL
    df = df.drop_duplicates(subset="SOURCEURL").reset_index(drop=True)

    print(f"{year}: {len(df)} unique NYT URLs in window {start}–{end}")
    return df


gdelt_data = {year: load_gdelt(year) for year in ELECTIONS_TO_RUN}

2020: 81 unique NYT URLs in window 20201103–20201105
2024: 6 unique NYT URLs in window 20241105–20241107


## 4. Selenium setup & NYT login

In [6]:
#A
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import undetected_chromedriver as uc

In [7]:
#B
def build_driver():
    options = uc.ChromeOptions()
    options.add_argument("--window-size=1920,1080")
    # NOT headless — undetected-chromedriver works best headed
    # headless mode is still detectable by NYT
    driver = uc.Chrome(options=options)
    return driver


def nyt_login(driver, email, password):
    wait = WebDriverWait(driver, 30)
    print("Navigating to NYT login...")
    driver.get("https://myaccount.nytimes.com/auth/login")
    time.sleep(5)  # let the page fully settle

    # Step 1: email
    email_field = wait.until(EC.presence_of_element_located((By.ID, "email")))
    email_field.clear()
    email_field.send_keys(email)

    continue_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "button[type='submit']")))
    continue_btn.click()
    time.sleep(4)

    # Step 2: password
    # NYT sometimes uses "password" and sometimes "myAccountPassword" depending on the flow
    # Try both
    try:
        password_field = wait.until(EC.presence_of_element_located((By.ID, "password")))
    except:
        password_field = wait.until(EC.presence_of_element_located((By.ID, "myAccountPassword")))

    password_field.clear()
    password_field.send_keys(password)

    login_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "button[type='submit']")))
    login_btn.click()
    time.sleep(5)

    print("Current URL after login:", driver.current_url)

    if "login" in driver.current_url.lower():
        raise RuntimeError("Login may have failed — still on login page.")
    print("Login successful.")


driver = build_driver()
nyt_login(driver, NYT_EMAIL, NYT_PASSWORD)

Navigating to NYT login...
Current URL after login: https://www.nytimes.com/?login=email&auth=login-email


RuntimeError: Login may have failed — still on login page.

## 5. Scraping function

In [8]:
#c
SKIP_PHRASES = {
    "Advertisement",
    "Supported by",
    "Send any friend a story",
    "As a subscriber, you have 10 gift articles to give each month. Anyone can read what you share.",
    "A version of this article appears in print",
    "Subscribe for $1/week.",
    "Thanks for reading The Times.",
}


def get_body(url: str, sleep_secs: int = 8) -> str:
    try:
        driver.get(url)
        time.sleep(sleep_secs)
        soup = BeautifulSoup(driver.page_source, "html.parser")

        container = (
            soup.find("section", {"name": "articleBody"})
            or soup.find("div", {"itemprop": "articleBody"})
            or soup.find("div", {"class": lambda c: c and "article-body" in " ".join(c)})
            or soup
        )

        article_text = ""
        for p in container.find_all("p"):
            text = p.get_text(separator=" ", strip=True)
            if not text or text in SKIP_PHRASES or text.startswith("By "):
                continue
            article_text += text + " "

        return article_text.strip()

    except Exception as e:
        print(f"  ERROR scraping {url}: {e}")
        return ""

## 6. Scrape all elections & save separately

In [9]:
results = {}

for year in ELECTIONS_TO_RUN:
    df = gdelt_data[year].copy()
    n = len(df)
    print(f"\n{'='*50}")
    print(f"Scraping {year} ({n} articles)")
    print(f"{'='*50}")

    bodies = []
    for i, row in df.iterrows():
        url = row["SOURCEURL"]
        print(f"  [{i+1}/{n}] {url}")
        body = get_body(url)
        bodies.append(body)
        status = f"{len(body)} chars" if body else "EMPTY — possible paywall/error"
        print(f"    → {status}")

    df["article_body"]  = bodies
    df["body_length"]   = df["article_body"].str.len()
    df["election_year"] = year

    # Save this election immediately (don't lose work if later elections crash)
    out_path = OUTPUT_PATHS[year]
    df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"\n  Saved {len(df)} rows → {out_path}")

    # Warn about likely failures
    empty = df[df["body_length"] < 100]
    if not empty.empty:
        print(f"  ⚠ {len(empty)} articles returned < 100 chars (check below):")
        for _, r in empty.iterrows():
            print(f"    {r['SOURCEURL']}")

    results[year] = df

print("\nAll done.")


Scraping 2020 (81 articles)
  [1/81] https://www.nytimes.com/2021/11/02/us/elections/minneapolis-reject-defund-police.html
    → 59913 chars
  [2/81] https://www.nytimes.com/2021/11/05/us/politics/project-veritas-investigation-ashley-biden-diary.html
    → 8237 chars
  [3/81] https://www.nytimes.com/2020/11/05/us/election-protests-portland-minnesota-arizona.html
    → 5909 chars
  [4/81] https://www.nytimes.com/2020/11/02/briefing/election-eve-polls-who-vienna.html
    → 7684 chars
  [5/81] https://www.nytimes.com/2020/11/04/us/politics/biden-is-closing-the-gap-on-trump-in-georgia.html
    → 1789 chars
  [6/81] https://www.nytimes.com/2020/11/02/us/politics/caregivers-in-pennsylvania-grapple-with-the-coronaviruss-pain-how-will-they-vote.html
    → 2183 chars
  [7/81] https://www.nytimes.com/interactive/2020/11/04/us/elections/florida-election-results-by-county.html
    → 15246 chars
  [8/81] https://www.nytimes.com/2020/11/03/us/politics/the-end-of-a-chaotic-campaign-election-day.html

## 7. Close the browser

In [10]:
driver.quit()

## 8. Summary across elections

In [11]:
for year, df in results.items():
    total     = len(df)
    succeeded = (df["body_length"] >= 100).sum()
    failed    = total - succeeded
    print(f"{year}: {succeeded}/{total} scraped successfully, {failed} failed/empty")

2020: 80/81 scraped successfully, 1 failed/empty
2024: 6/6 scraped successfully, 0 failed/empty
